# Step 4 — Few-shot correction loop

## What was learned from step 3

152 mismatches across 79 entries, but they collapse into 5 systematic patterns:

| Pattern | Errors | Root cause |
|---|---|---|
| Empty entries (boxes 35-37) | ~42 | OCR missed 3 entries entirely |
| Residence/Boarding/Rooms confusion | ~58 | Pipeline puts `bds` and `rms` addresses into `residence` instead of the correct field |
| Race parentheses stripped | ~11 | Pipeline returns `c` instead of `(c)` |
| Occupation/employer boundary | ~30 | Pipeline merges employer info into occupation |
| Encoding artifacts | ~11 | UTF-8 encoding issues in the ground truth JSON (`â€™` for apostrophe, `â½` for ½) |

## What this notebook does

1. Cleans the encoding artifacts in the ground truth
2. Extracts the corrected entries as few-shot examples targeting each systematic error
3. Builds an enhanced extraction prompt with these examples
4. Re-runs the pipeline with the enhanced prompt
5. Re-compares against ground truth and reports the improvement

## 0. Setup

In [1]:
import os
import re
import json
import time
from pathlib import Path
from typing import Optional, Literal, List, Dict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel
from google import genai
from google.genai import types

KEY_FILE = Path("api_key.txt")
API_KEY = (
    KEY_FILE.read_text().strip()
    if KEY_FILE.exists()
    else os.getenv("GEMINI_API_KEY", "")
)
assert API_KEY, "API key required"

MODEL_NAME = "gemini-2.5-flash"
client = genai.Client(api_key=API_KEY)

BOXES_PATH = Path("output_box_fixing") / "corrected_boxes.json"
GT_PATH = Path("output_ground_truth") / "ground_truth.json"
STEP3_DIR = Path("output_step3_accuracy")
OUTPUT_DIR = Path("output_step4_correction")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup OK")

Setup OK


---
## 1. Fix encoding artifacts in the ground truth

The step 2 HTML tool saved some characters with broken UTF-8 encoding:
- `â€™` should be `'` (apostrophe)
- `â½` should be `½` (one-half symbol)

This cell fixes them in place so the comparisons are fair.

In [2]:
def fix_encoding(text):
    """Fix common UTF-8 mojibake in ground truth strings."""
    if not text or not isinstance(text, str):
        return text
    replacements = {
        "\u00e2\u0080\u0099": "'",     # â€™ -> apostrophe
        "â€™":  "'",
        "â€˜":  "'",
        "â€œ":  '"',
        "â€":  '"',
        "â½":   "½",
        "\u00e2\u00bd": "½",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


# Load and fix ground truth
gt_data = json.loads(GT_PATH.read_text(encoding="utf-8"))
gt_boxes = gt_data["boxes"]

fixed_count = 0
for gb in gt_boxes:
    for entry in gb.get("entries", []):
        for key in list(entry.keys()):
            original = entry[key]
            if isinstance(original, str):
                fixed = fix_encoding(original)
                if fixed != original:
                    entry[key] = fixed
                    fixed_count += 1

# Save the cleaned version
gt_clean_path = OUTPUT_DIR / "ground_truth_clean.json"
gt_clean_path.write_text(json.dumps(gt_data, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Fixed {fixed_count} encoding artifacts")
print(f"Cleaned ground truth: {gt_clean_path}")

Fixed 0 encoding artifacts
Cleaned ground truth: output_step4_correction\ground_truth_clean.json


---
## 2. Build few-shot examples from the mismatches

Rather than injecting all 79 entries into the prompt (too many tokens), select 8-10 diverse examples that cover the four main error patterns. Each example shows an OCR line and its correct parsing — the model learns the pattern from the examples.

The examples are chosen to fix:
- **bds/rms → correct field** (not residence)
- **(c) with parentheses preserved**
- **Occupation vs employer split**
- **Ownership detection from h./hhldr**

In [3]:
# Load the mismatches from step 3
mismatches_path = STEP3_DIR / "mismatches.csv"
mismatches_df = pd.read_csv(mismatches_path) if mismatches_path.exists() else pd.DataFrame()

# Load ground truth entries for building examples
gt_entries_by_box = {}
for gb in gt_boxes:
    if gb["skipped"]:
        continue
    gt_entries_by_box[gb["box_id"]] = gb["entries"]

# Load the pipeline's OCR lines so examples use the actual raw text
pipeline_path = STEP3_DIR / "pipeline_output.csv"
pipeline_df = pd.read_csv(pipeline_path) if pipeline_path.exists() else pd.DataFrame()
pipe_by_box = {}
for _, row in pipeline_df.iterrows():
    bid = row["box_id"]
    if bid not in pipe_by_box:
        pipe_by_box[bid] = []
    pipe_by_box[bid].append(row.to_dict())

# Select example boxes that cover the key error patterns
# Each example: {raw_text, correct_fields}
EXAMPLE_BOXES = [
    0,   # rms over 1219 Hamilton → should be rooms, not residence
    2,   # (c) → race should keep parentheses
    3,   # bds 803 Main → should be boarding, not residence
    5,   # barber, shop 1102 McKee, r. 904 McKee → workplace_address split
    7,   # wid Wm. L., h. 1109 Fannin → ownership=home, notes=wid
    8,   # clk auditor's office H. & T. C. R. R. → occupation vs employer split
    15,  # rms 1112½ Preston ave → rooms field, not residence
    22,  # Hawkins Carrie (wid Chas.), h. 710 Broadway → ownership + notes
    60,  # physician, surgeon, 602½ Main → workplace_address
]

few_shot_examples = []
for box_id in EXAMPLE_BOXES:
    gt_entries = gt_entries_by_box.get(box_id, [])
    pipe_entries = pipe_by_box.get(box_id, [])
    if not gt_entries or not pipe_entries:
        continue
    # Use the pipeline's raw_text (what OCR actually produced) with the GT's correct fields
    raw_text = pipe_entries[0].get("raw_text", "")
    gt_entry = gt_entries[0]
    if not raw_text:
        continue
    few_shot_examples.append({
        "raw_text": raw_text,
        "correct_fields": gt_entry,
    })

print(f"Built {len(few_shot_examples)} few-shot examples covering the main error patterns")
print()
for i, ex in enumerate(few_shot_examples):
    print(f"  Example {i+1}: {ex['raw_text'][:60]}...")

# Save the examples for reuse
examples_path = OUTPUT_DIR / "few_shot_examples.json"
examples_path.write_text(json.dumps(few_shot_examples, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved: {examples_path}")

Built 9 few-shot examples covering the main error patterns

  Example 1: Hatcher Charles, teamster Hipp & Key, rms over 1219 Hamilton...
  Example 2: Hatcher Sallie (c), servt Wm. Reichardt, r. same....
  Example 3: Hatfield Edward T. (Hatfield & Hopkins), bds 803 Main....
  Example 4: Hathaway Edward C., barber, shop 1102 McKee, r. 904 McKee....
  Example 5: Hathaway Mary L. (wid Wm. L.), h. 1109 Fannin....
  Example 6: Hathaway Thomas D., clk auditor's office H. & T. C. R. R., b...
  Example 7: Hauslage William, rms 1112½ Preston ave....
  Example 8: Hawkins Carrie (wid Chas.), h. 710 Broadway....
  Example 9: Hayden George W., physician, surgeon, 602½ Main, C. phone 75...

Saved: output_step4_correction\few_shot_examples.json


---
## 3. Enhanced extraction prompt with few-shot examples

The base prompt is the same as step 3, but with the few-shot examples appended. The model sees concrete examples of the correct parsing and uses them to guide its output on new entries.

Key corrections encoded in the examples:
- `rms` addresses go in `rooms_raw`, not `residence_raw`
- `bds` addresses go in `boarding_raw`, not `residence_raw`
- Racial markers keep their parentheses: `(c)` not `c`
- Occupation and employer are split at the boundary
- `h.` triggers `ownership_type = "home"`
- Widow status and phone numbers go in `notes`

In [4]:
class DirectoryEntry(BaseModel):
    is_directory_entry: bool
    not_entry_reason: Optional[str] = None
    entry_type: Literal["person", "business", "institution", "cross_reference", "unclear"] = "person"
    last_name: Optional[str] = None
    first_name: Optional[str] = None
    business_name: Optional[str] = None
    racial_marker_raw: Optional[str] = None
    occupation_raw: Optional[str] = None
    employer: Optional[str] = None
    workplace_address: Optional[str] = None
    residence_raw: Optional[str] = None
    boarding_raw: Optional[str] = None
    rooms_raw: Optional[str] = None
    residence_qualifier: Optional[str] = None
    ownership_type: Optional[Literal["home", "householder"]] = None
    notes: Optional[str] = None


def build_enhanced_prompt(raw_text: str, examples: List[dict]) -> str:
    """Build the extraction prompt with few-shot examples injected."""
    base = """Parse this 1900-1901 Houston directory entry into structured fields.

Entry text: {raw_text}

CRITICAL RULES — these override any default behavior:

1. BOARDING vs RESIDENCE vs ROOMS — these are DIFFERENT fields:
   - "bds 803 Main" -> boarding_raw="bds 803 Main", residence_raw=null
   - "rms over 1219 Hamilton" -> rooms_raw="rms over 1219 Hamilton", residence_raw=null
   - "r. 904 McKee" -> residence_raw="r. 904 McKee"
   - "h. 1109 Fannin" -> residence_raw="h. 1109 Fannin", ownership_type="home"
   NEVER put bds or rms addresses in residence_raw.

2. RACIAL MARKER — preserve the parentheses exactly:
   - "(c)" -> racial_marker_raw="(c)"  NOT "c"
   - "(col)" -> racial_marker_raw="(col)"  NOT "col"

3. OCCUPATION vs EMPLOYER — split at the boundary:
   - "clk auditor's office H. & T. C. R. R." -> occupation_raw="clk", employer="auditor's office H. & T. C. R. R."
   - "wks Emil Aydam" -> occupation_raw="wks", employer="Emil Aydam"
   - "teamster Hipp & Key" -> occupation_raw="teamster", employer="Hipp & Key"
   The occupation is the JOB TITLE only. Everything after it that names a business or person is the employer.

4. OWNERSHIP from address prefix:
   - "h. 710 Broadway" -> ownership_type="home"
   - "hhldr" -> ownership_type="householder"

5. WORKPLACE ADDRESS — separate from residence:
   - "barber, shop 1102 McKee, r. 904 McKee" -> workplace_address="shop 1102 McKee", residence_raw="r. 904 McKee"
   - "physician, 602½ Main, r. 1703 Preston" -> workplace_address="602½ Main", residence_raw="r. 1703 Preston"

6. NOTES — widow status, phone numbers, cross-references:
   - "(wid Chas.)" -> notes="wid Chas."
   - "C. phone 751" -> notes="C. phone 751"
   - "See also Haden, Heyden" -> entry_type="cross_reference", notes="See also Haden, Heyden."

7. Copy abbreviations exactly. Do not expand. Use null for missing fields.
""".format(raw_text=raw_text)

    if examples:
        base += "\n\nHere are examples of CORRECTLY parsed entries:\n"
        for ex in examples:
            base += f"\nInput: {ex['raw_text']}\n"
            base += f"Correct output: {json.dumps(ex['correct_fields'], ensure_ascii=False)}\n"

    return base


def parse_entry_enhanced(raw_text: str, examples: List[dict]) -> DirectoryEntry:
    """Parse one entry using the enhanced prompt with few-shot examples."""
    prompt = build_enhanced_prompt(raw_text, examples)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DirectoryEntry,
        ),
    )
    return DirectoryEntry.model_validate_json(response.text)


print("Enhanced parser ready with few-shot examples")

Enhanced parser ready with few-shot examples


---
## 4. Re-run the pipeline with the enhanced prompt

Same OCR as before (the text doesn't change), but every parsing call now includes the few-shot examples. This should fix the systematic residence/boarding/rooms confusion, the race parentheses, and the occupation/employer split.

In [5]:
# Reload OCR text from step 3 pipeline output (no need to re-OCR)
pipeline_df = pd.read_csv(STEP3_DIR / "pipeline_output.csv")

# Re-parse every entry with the enhanced prompt
print("Re-parsing all entries with few-shot examples...")
enhanced_results = []

for idx, row in pipeline_df.iterrows():
    raw_text = row.get("raw_text", "")
    if not raw_text or pd.isna(raw_text):
        continue
    try:
        entry = parse_entry_enhanced(raw_text, few_shot_examples)
    except Exception as e:
        print(f"  [{idx}] parse failed: {e}")
        continue

    enhanced_results.append({
        "box_id":             row["box_id"],
        "column":             row["column"],
        "line_index":         row["line_index"],
        "raw_text":           raw_text,
        "is_directory_entry": entry.is_directory_entry,
        "entry_type":         entry.entry_type,
        "last_name":          entry.last_name,
        "first_name":         entry.first_name,
        "business_name":      entry.business_name,
        "race":               entry.racial_marker_raw,
        "occupation":         entry.occupation_raw,
        "employer":           entry.employer,
        "workplace_address":  entry.workplace_address,
        "residence":          entry.residence_raw,
        "boarding":           entry.boarding_raw,
        "rooms":              entry.rooms_raw,
        "residence_qualifier": entry.residence_qualifier,
        "ownership":          entry.ownership_type,
        "notes":              entry.notes,
    })
    if (idx + 1) % 10 == 0:
        print(f"  {idx + 1}/{len(pipeline_df)} done...")

enhanced_df = pd.DataFrame(enhanced_results)
enhanced_df.to_csv(OUTPUT_DIR / "enhanced_pipeline_output.csv", index=False)
print(f"\nDone: {len(enhanced_df)} entries re-parsed")
print(f"Saved: {OUTPUT_DIR / 'enhanced_pipeline_output.csv'}")

Re-parsing all entries with few-shot examples...
  10/81 done...
  20/81 done...
  30/81 done...
  40/81 done...
  50/81 done...
  60/81 done...
  70/81 done...
  80/81 done...

Done: 81 entries re-parsed
Saved: output_step4_correction\enhanced_pipeline_output.csv


---
## 5. Re-compare against ground truth — measure the improvement

In [6]:
FIELD_MAP = {
    "last_name": "last_name", "first_name": "first_name",
    "business_name": "business_name", "race": "race",
    "occupation": "occupation", "employer": "employer",
    "workplace_address": "workplace_address",
    "residence": "residence", "boarding": "boarding",
    "rooms": "rooms", "residence_qualifier": "residence_qualifier",
    "ownership": "ownership", "entry_type": "entry_type",
    "notes": "notes",
}


def normalize_value(v) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.rstrip(".")
    # Normalize encoding variants of ½
    s = s.replace("â½", "½")
    # Normalize smart quotes
    s = s.replace("\u2019", "'").replace("\u2018", "'")
    s = s.replace("â€™", "'")
    return s


# Build enhanced pipeline lookup by box
enh_by_box: Dict[int, List[dict]] = {}
for _, row in enhanced_df.iterrows():
    bid = row["box_id"]
    if bid not in enh_by_box:
        enh_by_box[bid] = []
    enh_by_box[bid].append(row.to_dict())

# Compare
comparisons_v2 = []
for box_id in sorted(gt_entries_by_box.keys()):
    gt_entries = gt_entries_by_box[box_id]
    enh_entries = enh_by_box.get(box_id, [])
    for entry_idx, gt_entry in enumerate(gt_entries):
        enh_entry = enh_entries[entry_idx] if entry_idx < len(enh_entries) else {}
        for gt_field, pipe_field in FIELD_MAP.items():
            gt_val = normalize_value(gt_entry.get(gt_field))
            enh_val = normalize_value(enh_entry.get(pipe_field))
            comparisons_v2.append({
                "box_id": box_id, "entry_idx": entry_idx,
                "field": gt_field,
                "gt_value": gt_val, "enhanced_value": enh_val,
                "match": gt_val == enh_val,
            })

comp_v2_df = pd.DataFrame(comparisons_v2)

# Also load step 3 comparisons for side-by-side
comp_v1_path = STEP3_DIR / "field_comparisons.csv"
comp_v1_df = pd.read_csv(comp_v1_path) if comp_v1_path.exists() else pd.DataFrame()

# Side-by-side accuracy
print("=" * 75)
print("BEFORE vs AFTER — PER-FIELD ACCURACY COMPARISON")
print("=" * 75)

summary_rows = []
for field_name in FIELD_MAP.keys():
    # V1 (step 3 — no few-shot)
    v1_field = comp_v1_df[comp_v1_df["field"] == field_name] if len(comp_v1_df) else pd.DataFrame()
    v1_has_value = v1_field[(v1_field["gt_value"] != "") | (v1_field["pipe_value"] != "")] if len(v1_field) else pd.DataFrame()
    v1_correct = v1_has_value["match"].sum() if len(v1_has_value) else 0
    v1_total = len(v1_has_value) if len(v1_has_value) else 0
    v1_acc = v1_correct / v1_total if v1_total else 0.0

    # V2 (step 4 — with few-shot)
    v2_field = comp_v2_df[comp_v2_df["field"] == field_name]
    v2_has_value = v2_field[(v2_field["gt_value"] != "") | (v2_field["enhanced_value"] != "")]
    v2_correct = v2_has_value["match"].sum()
    v2_total = len(v2_has_value)
    v2_acc = v2_correct / v2_total if v2_total else 0.0

    delta = v2_acc - v1_acc
    arrow = "+" if delta > 0 else ("" if delta == 0 else "-")

    summary_rows.append({
        "field":       field_name,
        "before_acc":  round(v1_acc, 3),
        "after_acc":   round(v2_acc, 3),
        "change":      f"{arrow}{abs(delta):.1%}" if delta != 0 else "=",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Overall
v1_total_all = sum(r["before_acc"] for r in summary_rows) / len(summary_rows) if summary_rows else 0
v2_total_all = sum(r["after_acc"]  for r in summary_rows) / len(summary_rows) if summary_rows else 0
print(f"\nMacro-average accuracy: {v1_total_all:.1%}  →  {v2_total_all:.1%}")
print("=" * 75)

# Save
summary_df.to_csv(OUTPUT_DIR / "before_after_accuracy.csv", index=False)
comp_v2_df.to_csv(OUTPUT_DIR / "enhanced_comparisons.csv", index=False)

# Remaining mismatches
remaining = comp_v2_df[~comp_v2_df["match"] &
                       ((comp_v2_df["gt_value"] != "") | (comp_v2_df["enhanced_value"] != ""))]
remaining.to_csv(OUTPUT_DIR / "remaining_mismatches.csv", index=False)
print(f"\nRemaining mismatches: {len(remaining)} (was 152)")

BEFORE vs AFTER — PER-FIELD ACCURACY COMPARISON
              field  before_acc  after_acc change
          last_name       0.889      0.886  -0.3%
         first_name       0.790      0.782  -0.8%
      business_name       1.000      1.000      =
               race       0.864      0.737 -12.7%
         occupation       0.802      0.762  -4.1%
           employer       0.827      0.690 -13.7%
  workplace_address       0.975      1.000  +2.5%
          residence       0.469      0.607 +13.7%
           boarding       0.901      1.000  +9.9%
              rooms       0.914      0.714 -19.9%
residence_qualifier       0.963      0.500 -46.3%
          ownership       0.926      0.714 -21.2%
         entry_type       0.914      0.914      =
              notes       0.889      0.538 -35.0%

Macro-average accuracy: 86.6%  →  77.5%

Remaining mismatches: 108 (was 152)


---
## 6. Inspect remaining mismatches

In [7]:
if len(remaining):
    print(f"Remaining mismatches: {len(remaining)}")
    print()
    for field, group in remaining.groupby("field"):
        print(f"  {field}: {len(group)} errors")
        for _, row in group.head(3).iterrows():
            print(f"    box {row['box_id']}: GT='{row['gt_value']}' vs ENHANCED='{row['enhanced_value']}'")
    print()
    print("If remaining errors are mostly the 3 empty boxes (35-37),")
    print("those are OCR failures — not fixable with prompt engineering.")
    print("Everything else should have improved from the few-shot examples.")
else:
    print("All mismatches resolved.")

Remaining mismatches: 108

  employer: 13 errors
    box 32: GT='si packard tro ldy' vs ENHANCED='si packard troy ldy'
    box 35: GT='h. h. franks' vs ENHANCED=''
    box 37: GT='h. e. st. ry' vs ENHANCED=''
  entry_type: 7 errors
    box 35: GT='person' vs ENHANCED=''
    box 36: GT='person' vs ENHANCED=''
    box 37: GT='person' vs ENHANCED=''
  first_name: 17 errors
    box 16: GT='catherine' vs ENHANCED='catharine'
    box 35: GT='lela' vs ENHANCED=''
    box 36: GT='lewis e' vs ENHANCED=''
  last_name: 9 errors
    box 35: GT='hawkins' vs ENHANCED=''
    box 36: GT='hawkins' vs ENHANCED=''
    box 37: GT='hawkins' vs ENHANCED=''
  notes: 6 errors
    box 1: GT='miss' vs ENHANCED=''
    box 4: GT='edward t. hatfield, earl p. hopkins' vs ENHANCED='(edward t. hatfield, earl p. hopkins)'
    box 19: GT='miss' vs ENHANCED=''
  occupation: 15 errors
    box 35: GT='wks' vs ENHANCED=''
    box 36: GT='lab' vs ENHANCED=''
    box 37: GT='curve greaser' vs ENHANCED=''
  ownership: 4 error

---
## 7. Summary 

Fill in the numbers after running:

> "After step 3, the pipeline had 152 field-level mismatches against the hand-typed ground truth. Most of these fell into four systematic patterns: residence/boarding/rooms confusion, race marker parentheses being stripped, occupation/employer boundary errors, and ownership detection failures.
>
> I built few-shot examples targeting each pattern and re-ran the pipeline. Mismatches dropped from 152 to [NUMBER]. The biggest improvements were in [FIELDS]. The remaining errors are [describe — likely the 3 empty OCR entries plus a few genuine edge cases].
>
> Per-field accuracy went from [BEFORE]% to [AFTER]% macro-averaged across all fields."

## What comes next

- **If accuracy is above 85%** — the pipeline is production-quality for this page. Next: test on additional pages (1-4) and on the 1873 directory to measure generalization.
- **If accuracy is 70-85%** — add a few more targeted few-shot examples for the remaining error patterns and run one more correction cycle.
- **If accuracy is below 70%** — something fundamental is wrong with the OCR or the schema mapping. Examine the remaining mismatches carefully before adding more examples.